In [8]:
import pandas as pd
import joblib

def load_and_test_prediction(data):
    try:
        model = joblib.load("performance_model.pkl")
        encoders = joblib.load("feature_encoders.pkl")
        target_encoder = joblib.load("target_encoder.pkl")

        row = {}

        for col in ["vehicle_type", "service_type"]:
            le = encoders[col]
            val = data[col] if data[col] in le.classes_ else le.classes_[0]
            row[col] = le.transform([val])[0]

        row["experience_years"] = data["experience_years"]
        row["job_count"] = data.get("job_count", 0)
        row["work_success_rate_num"] = data.get("work_success_rate_num", 100.0)
        row["customer_rating"] = data.get("customer_rating", 4.0)
        row["historical_speed_ratio"] = data.get("historical_speed_ratio", 1.0)

        X = pd.DataFrame([row])

        expected_columns = ['vehicle_type', 'service_type', 'experience_years', 'job_count',
                            'work_success_rate_num', 'customer_rating', 'historical_speed_ratio']
        X = X[expected_columns]

        pred_enc = model.predict(X)[0]
        pred_label = target_encoder.inverse_transform([pred_enc])[0]

        return pred_label
    except Exception as e:
        return f"Error: {str(e)}"

test_technician_data = {
    "vehicle_type": "Car",
    "service_type": "Body Wash",
    "experience_years": 2,
    "job_count": 2,
    "work_success_rate_num": 40.0,
    "customer_rating": 0.0,
    "historical_speed_ratio": 0.80
}

result = load_and_test_prediction(test_technician_data)
print("--------------------------------------------------")
print(f"This employee's predicted level: {result}")
print("--------------------------------------------------")

--------------------------------------------------
This employee's predicted level: MEDIUM
--------------------------------------------------


In [5]:
import pandas as pd
import glob
import os

files = glob.glob('*.xlsx')
print("Excel files:", files)

if 'technician_ml_dataset.xlsx' in files:
    df = pd.read_excel('technician_ml_dataset.xlsx')
    # Clean the work_success_rate
    df['work_success_rate_num'] = df['work_success_rate'].astype(str).str.replace('%', '').astype(float)

    # Check correlation
    print("Correlation:")
    print(df[['job_count', 'work_success_rate_num']].corr())

    # Show a sample
    print("\nSample:")
    print(df[['job_count', 'work_success_rate', 'work_success_rate_num']].head(10))

    # Check if work_success_rate makes mathematical sense based on job_count
    # e.g., if job_count is 23, success rate could be (successful_jobs / 23) * 100
    # Let's see if successful_jobs is an integer
    df['calculated_successful_jobs'] = (df['work_success_rate_num'] / 100) * df['job_count']
    print("\nCalculated successful jobs (should be very close to integers):")
    print(df[['job_count', 'work_success_rate_num', 'calculated_successful_jobs']].head(15))

Excel files: ['technician_ml_dataset.xlsx']
Correlation:
                       job_count  work_success_rate_num
job_count               1.000000               0.013668
work_success_rate_num   0.013668               1.000000

Sample:
   job_count  work_success_rate  work_success_rate_num
0         23              0.870                  0.870
1         25              0.920                  0.920
2         41              0.829                  0.829
3         26              0.923                  0.923
4         36              0.861                  0.861
5         43              0.977                  0.977
6         24              0.917                  0.917
7         34              0.824                  0.824
8         39              0.821                  0.821
9         25              0.920                  0.920

Calculated successful jobs (should be very close to integers):
    job_count  work_success_rate_num  calculated_successful_jobs
0          23                  0

In [6]:
import pandas as pd
import numpy as np

df = pd.read_excel('technician_ml_dataset.xlsx')

if 'customer_satisf...' in df.columns:
    df.rename(columns={'customer_satisf...': 'customer_satisfaction'}, inplace=True)
if 'avg_actual_time...' in df.columns:
    df.rename(columns={'avg_actual_time...': 'avg_actual_time_hrs'}, inplace=True)

df['work_success_rate_num'] = df['work_success_rate'].astype(str).str.replace('%', '').astype(float)
df['work_success_rate_num'] = df['work_success_rate_num'].apply(lambda x: x * 100 if x <= 1.0 else x)

rating_map = {'High': 4.8, 'Medium': 4.0, 'Low': 3.0}
df['customer_rating'] = df['customer_satisfaction'].map(rating_map)

def create_labels(speed):
    if speed >= 1.05:
        return 'HIGH'
    elif speed < 0.95:
        return 'LOW'
    else:
        return 'MEDIUM'

df['Performance_Label'] = df['speed_ratio'].apply(create_labels)

technician_avg_speed = df.groupby('technician_id')['speed_ratio'].mean().reset_index()
technician_avg_speed.rename(columns={'speed_ratio': 'historical_speed_ratio'}, inplace=True)
df = df.merge(technician_avg_speed, on='technician_id', how='left')

columns_to_drop = [
    'center_id', 'technician_id', 'month', 'work_success_rate',
    'customer_satisfaction', 'expected_time_hrs', 'avg_actual_time_hrs',
    'speed_ratio'
]
df_prepared = df.drop(columns=columns_to_drop)

df_prepared.to_csv('prepared_technician_dataset.csv', index=False)

print("The dataset was successfully prepared!\n")
print("How the new labels are distributed:")
print(df_prepared['Performance_Label'].value_counts())
print("\nThe final columns fed into the ML model:")
print(df_prepared.columns.tolist())

The dataset was successfully prepared!

How the new labels are distributed:
Performance_Label
MEDIUM    1278
LOW        882
HIGH       720
Name: count, dtype: int64

The final columns fed into the ML model:
['vehicle_type', 'service_type', 'experience_years', 'job_count', 'work_success_rate_num', 'customer_rating', 'Performance_Label', 'historical_speed_ratio']


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

df = pd.read_csv('prepared_technician_dataset.csv')

X = df.drop('Performance_Label', axis=1)
y = df['Performance_Label']

feature_encoders = {}
for col in ['vehicle_type', 'service_type']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    feature_encoders[col] = le

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("\n--- Model Performance (Classification Report) ---")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

joblib.dump(model, 'performance_model.pkl')
joblib.dump(feature_encoders, 'feature_encoders.pkl')
joblib.dump(target_encoder, 'target_encoder.pkl')

print("\nDone! The three files—performance_model.pkl, feature_encoders.pkl, and target_encoder.pkl—were successfully saved..")


--- Model Performance (Classification Report) ---
              precision    recall  f1-score   support

        HIGH       1.00      1.00      1.00       126
         LOW       0.90      0.97      0.93       181
      MEDIUM       0.98      0.93      0.95       269

    accuracy                           0.96       576
   macro avg       0.96      0.97      0.96       576
weighted avg       0.96      0.96      0.96       576


Done! The three files—performance_model.pkl, feature_encoders.pkl, and target_encoder.pkl—were successfully saved..
